# 03 - Bounding-Box & Lesion Analysis

Analyse the annotated lesions and their bounding boxes, then **simulate** what happens to lesion size when images are letterbox-resized to the candidate YOLO input resolutions 640 / 960 / 1024. This is the evidence for the question *"is 640 enough to detect acne lesions?"*

> Read-only.

In [1]:
# Make the src/ package importable from the notebooks/ folder (CPU-only, no install).
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
pd.set_option('display.max_columns', 30)

from acnevision.data import (
    eda_data, eda_images, eda_severity, eda_lesions, eda_boxes, eda_viz,
)

FIG = Path.cwd().parent / 'outputs' / 'figures'


In [2]:
# Load once: per-image table (+ real image sizes) and the per-box table.
df = eda_data.load_dataframe()
df = eda_data.attach_image_sizes(df)
boxes = eda_data.explode_boxes(df)
print('images:', len(df), '| lesion boxes:', len(boxes))
df.head(3)


images: 1457 | lesion boxes: 18983


,image_file,file_name,level,severity,lesion_count_label,num_lesions_annotated,lesion_boxes,split,fold_0_split,fold_1_split,fold_2_split,fold_3_split,fold_4_split,num_boxes,image_path,width,height,mode,img_format,missing,corrupted,aspect_ratio
0,acne_00000.jpg,levle0_0.jpg,0,mild,3,3,"{'xmin': [223, 217, 186], 'ymin': [290, 250, 5...",train,train,train,train,test,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,525,700,RGB,JPEG,False,False,0.750000
1,acne_00001.jpg,levle0_1.jpg,0,mild,3,3,"{'xmin': [394, 654, 930], 'ymin': [933, 2160, ...",train,train,train,test,train,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,3112,3456,RGB,JPEG,False,False,0.900463
2,acne_00002.jpg,levle0_10.jpg,1,moderate,20,20,"{'xmin': [357, 394, 314, 316, 265, 170, 203, 9...",train,train,train,train,test,train,20,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,478,460,RGB,JPEG,False,False,1.039130


In [3]:
# Per-box metrics: pixel width/height/area/aspect + image-normalized sizes.
metrics = eda_boxes.box_metrics(boxes, df)
metrics.head(3)

,image_file,severity,level,xmin,ymin,xmax,ymax,img_width,img_height,box_width,box_height,box_area,box_aspect,norm_width,norm_height,norm_area
0,acne_00000.jpg,mild,0,223,290,251,306,525,700,28,16,448,1.750000,0.053333,0.022857,0.001219
1,acne_00000.jpg,mild,0,217,250,239,276,525,700,22,26,572,0.846154,0.041905,0.037143,0.001556
2,acne_00000.jpg,mild,0,186,514,209,536,525,700,23,22,506,1.045455,0.043810,0.031429,0.001377


## 4. Lesion analysis

In [4]:
eda_lesions.lesion_count_summary(df)

{'total_lesions': 18983,
 'num_images': 1457,
 'mean_per_image': 13.03,
 'median_per_image': 7.0,
 'min_per_image': 1,
 'max_per_image': 65,
 'images_with_zero_boxes': 0}

In [5]:
eda_lesions.lesions_by_severity(df)

,severity,count,mean,median,min,max,sum
0,mild,513,2.10,2.0,1,5,1079
1,moderate,633,8.96,8.0,6,20,5670
2,severe,182,26.08,25.0,21,42,4747
3,very_severe,129,58.04,58.0,51,65,7487


In [6]:
eda_lesions.top_images_by_lesions(df, 10)

,image_file,severity,num_boxes,num_lesions_annotated
0,acne_01416.jpg,very_severe,65,65
1,acne_01417.jpg,very_severe,65,65
2,acne_01422.jpg,very_severe,65,65
3,acne_01342.jpg,very_severe,65,65
4,acne_01339.jpg,very_severe,65,65
5,acne_01371.jpg,very_severe,65,65
6,acne_01370.jpg,very_severe,65,65
7,acne_01377.jpg,very_severe,65,65
8,acne_01440.jpg,very_severe,65,65
9,acne_01325.jpg,very_severe,65,65


In [7]:
eda_viz.plot_lesion_count_hist(df, FIG / 'lesions_per_image_hist.png')
print('saved lesions-per-image histogram')

saved lesions-per-image histogram


### Count-field consistency
`num_boxes` (stored boxes) vs `num_lesions_annotated` vs `lesion_count_label`.

In [8]:
mm = eda_lesions.count_field_mismatches(df)
print('rows where the count fields disagree:', len(mm))
mm.head()

rows where the count fields disagree: 21


,image_file,severity,num_boxes,num_lesions_annotated,lesion_count_label
0,acne_01322.jpg,very_severe,65,65,58
1,acne_01323.jpg,very_severe,58,58,56
2,acne_01333.jpg,very_severe,57,57,58
3,acne_01335.jpg,very_severe,62,62,52
4,acne_01349.jpg,very_severe,57,57,60


## 5. Bounding-box size analysis (original resolution)

In [9]:
eda_boxes.box_metric_summary(metrics)

{'num_boxes': 18983,
 'box_width_px': {'min': 4.0, 'median': 93.0, 'mean': 93.1836, 'max': 514.0},
 'box_height_px': {'min': 4.0, 'median': 99.0, 'mean': 93.1105, 'max': 491.0},
 'box_area_px2': {'min': 16.0,
  'median': 10332.0,
  'mean': 12293.6399,
  'max': 241081.0},
 'box_aspect': {'min': 0.2159, 'median': 1.0, 'mean': 1.0535, 'max': 4.6923},
 'norm_width': {'min': 0.0018,
  'median': 0.0443,
  'mean': 0.0465,
  'max': 0.1875},
 'norm_height': {'min': 0.0025,
  'median': 0.0422,
  'mean': 0.0439,
  'max': 0.1504}}

In [10]:
eda_boxes.size_buckets(metrics)   # COCO small/medium/large on pixel area

,size,count,percentage
0,small,5140,27.08
1,medium,3955,20.83
2,large,9888,52.09


In [11]:
eda_boxes.invalid_boxes(metrics)  # empty => every box is geometrically valid

,image_file,xmin,ymin,xmax,ymax,box_width,box_height,img_width,img_height


In [12]:
eda_viz.plot_box_size_hist(metrics, FIG / 'box_size_hist.png')
print('saved box-size histograms')

saved box-size histograms


## Resize simulation: is 640 enough?
Letterbox scale = target / max(image_width, image_height). Every box is scaled by the same factor; we report the median lesion side, the % that fall into the COCO *small* bucket, and the % that become too tiny (<8 px) to detect.

In [13]:
sim = eda_boxes.simulate_resize(metrics, targets=(640, 960, 1024))
sim

,target_px,median_box_side_px,median_box_area_px2,pct_tiny_lt_8px,pct_small,pct_medium,pct_large
0,640,22.50,615.84,0.84,82.50,17.50,0.00
1,960,33.75,1385.65,0.14,29.76,70.22,0.02
2,1024,36.00,1576.56,0.13,22.59,77.34,0.07


In [14]:
eda_viz.plot_resize_simulation(sim, FIG / 'resize_simulation_tiny.png')
print('saved resize-simulation chart')

saved resize-simulation chart


**Takeaways:** at **640 px** the median lesion is only ~22 px and ~82% of lesions fall in the COCO *small* bucket - detectable but hard. At **960 px** the small share drops to ~30% (median ~34 px); **1024** adds little beyond 960. Recommendation: train at **960** (use 1024 only if recall on small lesions is still short). See `docs/acne04_eda_report.md` for the full YOLO guidance.